# Wind shear estimation from MERRA Data

* RESKit will vertically adjust wind speeds to account for wind shear
* Uses a logarithmic wind profile approach
* Requires estimating roughness length, and having a "known" wind speed

In [ ]:
import reskit as rk

import numpy as np
import matplotlib.pyplot as plt

%matplotlib inline

In [ ]:
# Create a weather source, and load data
src = rk.weather.MerraSource(rk.TEST_DATA["merra-like"], bounds=[5, 49, 7, 52], verbose=False)
src.sload_elevated_wind_speed()

# Get an adjusted time series for a specific location

In [ ]:
# Extract raw wind speed and GHI data
location = (6.0, 50.5)

windspeeds = src.get("elevated_wind_speed", locations=location)
print(windspeeds.head())

In [ ]:
# Estimate roughness length at the location from a land-cover dataset
#  - In this example, Corine Land Cover will be used
#  - in this case, roughness lengths are taken from the suggestions by Silva et al.
#    "Roughness Length Classification Of Corine Land Cover Classes". 2007

roughness = rk.wind.roughness_from_clc(clc_path=rk.TEST_DATA["clc-aachen_clipped.tif"], loc=location)

print("Estimated roughness is {}m".format(roughness))

In [ ]:
# Apply log-law projection
projected_windspeed = rk.wind.apply_logarithmic_profile_projection(
    measured_wind_speed=windspeeds,
    measured_height=50,  # The MERRA dataset offers windspeeds at 50m
    target_height=120,  # Assuming we want to project up to 120m
    roughness=roughness,
)

print(projected_windspeed.head())

# Get adjusted time series for multiple locations at once

In [ ]:
# Extract raw wind speed and GHI data
locations = [(6.25, 51.0), (6.50, 51.0), (6.25, 50.75)]

windspeeds = src.get("elevated_wind_speed", locations=locations)
windspeeds.head()

In [ ]:
roughness = rk.wind.roughness_from_clc(
    clc_path=rk.TEST_DATA["clc-aachen_clipped.tif"],
    loc=locations,
)

print("Estimated roughnesses are:")
print("  At {}: {}m".format(str(locations[0]), roughness[0]))
print("  At {}: {}m".format(str(locations[1]), roughness[1]))
print("  At {}: {}m".format(str(locations[2]), roughness[2]))

In [ ]:
projected_windspeed = rk.wind.apply_logarithmic_profile_projection(
    measured_wind_speed=windspeeds,
    measured_height=50,  # The MERRA dataset offers windspeeds at 50m
    target_height=120,  # Assuming we want to project up to 120m
    roughness=np.array(roughness),
)

projected_windspeed.head()